In [ ]:
# --- 1) Parameters ---
# Single file copy example. For multiple files, you can loop through the list of files and repeat the process.
# Since no S3 access code and Access Secret is giving, we are following this apporach. We are using the presigned URL to download the file and then we will upload it to lakehouse files.
presigned_url = r"""https://****.s3.ap-south-1.amazonaws.com/paraquet_backup/events_2025_01.parquet?response-content-disposition=inline&X-Amz-Content-Sha256=UNSIGNED-PAYLOAD&X-Amz-Security-Token=***"""
lakehouse_file_api_path = "/lakehouse/default/Files/RawFilesFromS3/events_2025_01.parquet"   # notebook mount path
lakehouse_spark_path = "Files/RawFilesFromS3/events_2025_01.parquet"                          # Spark relative path

# --- 2) Download (stream) the file into Lakehouse Files ---
import os
import requests

os.makedirs(os.path.dirname(lakehouse_file_api_path), exist_ok=True)

with requests.get(presigned_url, stream=True, timeout=300) as r:
    r.raise_for_status()
    with open(lakehouse_file_api_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                f.write(chunk)

print(f"Downloaded to: {lakehouse_file_api_path}")

StatementMeta(, 605db081-1318-4b2f-be7a-84590ab03ae3, 3, Finished, Available, Finished, False)

Downloaded to: /lakehouse/default/Files/RawFilesFromS3/events_2025_01.parquet


In [ ]:
# --- 1) Parameters ---
# This will copy all files under the specified S3 prefix to the target Lakehouse Files directory, preserving folder structure. 
# This example uses Hadoop's FileSystem API for efficient copying, which is suitable for large files and many files. Adjust the bucket, prefix, and target directory as needed.
# This will copy files in parallel using the underlying Hadoop S3A connector, which can significantly speed up the process compared to a single-threaded approach.
# This also creates checksum files (.crc) for each copied file, which can be used for integrity verification later if needed.
bucket = "XXXX"  # your S3 bucket name
prefix = "paraquet_backup/full_backup/events/"   # your "folder" (prefix) in S3

s3_prefix_path = f"s3a://{bucket}/{prefix}"
lakehouse_target_dir = "file:/lakehouse/default/Files/RawFilesFromS3/events/"  # target folder in Lakehouse Files

# --- 2) Configure S3A credentials (DO NOT hardcode real secrets in notebooks) ---
# Prefer setting these via environment variables or Fabric secret injection
import os

AWS_ACCESS_KEY_ID     = "xxx"  # your AWS Access Key ID
AWS_SECRET_ACCESS_KEY = "xxxx"  # your AWS Secret Access Key
AWS_SESSION_TOKEN     = "xxxx//////////wEaDF91fcxhKG/c+hmpcSKLAkeJwdiNJrp5drcRxFgXMdLjA2hEiFSKaKZAf3r1KlPAApRYVbqkRTpnmM/62Jnc+NgMCCTkOJ4aaGPb37OjmwwcJGuRHlyRctGxe3943jRwTD6ERAtKtUfd2VqI8F4IJaTy1rAQjoQnEnwlSv9fGIamAHWymkBSiA6VnDyeB48RQ3H04B/Pv0db7dQeHB6u5wabAYwGw9jjXq8Q9v5rrXBgeZgdlslTuF3RUl1DYA/dMckLw1XDLFvvQoAG+1M8uNfddiReZxhK/GpVAjJavBxBdGcGQhpe/Nakw38In/Zq+rCCmsnGc5a4dtRGDD6StOyDodt8dtP16snHWaDzVWkz+Oscz73aE+BFPijkndXMBjIpOJpy0I6UkDpHjZMTlwFEjR4//acGqyNhNdkznZaQJDpDnYlWhCkiHKc="
AWS_REGION            = "ap-south-1"

hconf = spark.sparkContext._jsc.hadoopConfiguration()
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hconf.set("fs.s3a.endpoint", f"s3.{AWS_REGION}.amazonaws.com")
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "true")

hconf.set("fs.s3a.access.key", AWS_ACCESS_KEY_ID)
hconf.set("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)

# If using temporary credentials (ASIA...), session token is mandatory
if AWS_SESSION_TOKEN:
    hconf.set("fs.s3a.session.token", AWS_SESSION_TOKEN)

# Optional stability/perf knobs
hconf.set("fs.s3a.connection.maximum", "200")
hconf.set("fs.s3a.attempts.maximum", "10")
hconf.set("fs.s3a.paging.maximum", "1000")

# --- 3) Copy objects from S3 prefix to Lakehouse Files (byte-for-byte) ---
jvm = spark.sparkContext._jvm
Path = jvm.org.apache.hadoop.fs.Path
FileSystem = jvm.org.apache.hadoop.fs.FileSystem

src_fs = FileSystem.get(Path(s3_prefix_path).toUri(), hconf)
dst_fs = FileSystem.get(Path(lakehouse_target_dir).toUri(), hconf)

# Ensure destination directory exists
dst_fs.mkdirs(Path(lakehouse_target_dir))

# List source files and copy
it = src_fs.listFiles(Path(s3_prefix_path), True)  # True = recursive
count = 0

while it.hasNext():
    status = it.next()
    src_path = status.getPath().toString()

    # Compute relative path under the prefix to preserve folder structure
    rel = src_path.replace(s3_prefix_path, "").lstrip("/")
    dst_path = (lakehouse_target_dir.rstrip("/") + "/" + rel)

    # Create parent directories
    parent = "/".join(dst_path.split("/")[:-1])
    dst_fs.mkdirs(Path(parent))

    # Copy file bytes
    jvm.org.apache.hadoop.fs.FileUtil.copy(src_fs, Path(src_path), dst_fs, Path(dst_path), False, hconf)
    count += 1

print(f"✅ Copied {count} file(s) from {s3_prefix_path} to {lakehouse_target_dir}")